# Urban Flood — Model 2 / NodeType 2 dataset builder + CatBoost + LightGBM + XGBoost ensemble


In [1]:
from __future__ import annotations

import gc
import re
import time
import warnings
import ctypes
from collections import defaultdict
from pathlib import Path
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigsh

import numpy as np
import pandas as pd
import pandas.api.types
from catboost import CatBoostRegressor
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

In [2]:
# =========================================================
# CONFIG
# =========================================================
DATA_ROOT = Path("/kaggle/input/datasets/bulbazavril/fullds/Models")
FEATURE_STORE_DIR = Path("./feature_store_m2_nt2")
FORCE_REBUILD_FEATURE_STORE = False

KEY_COLS = ["model_id", "event_id", "node_type", "node_id"]
TARGET_COL = "water_level"
SUBMISSION_COLS = ["row_id"] + KEY_COLS + [TARGET_COL]

VALID_EVENTS_PER_MODEL = 15
RANDOM_SEED = 42
MIN_PRED_TIMESTEP = 10
WARMUP_STEPS = 10
USE_TARGET_DELTA = False

USE_SPECTRAL_EMBED = False
SPECTRAL_DIM = 8
USE_AUX_TARGETS = False

print("Config ready (RAM-optimized).")

Config ready (RAM-optimized).


In [3]:
# =========================================================
# METRIC
# =========================================================
STD_DEV_DICT = {
    (1, 1): 16.877747, (1, 2): 14.378797,
    (2, 1): 3.191784,  (2, 2): 2.727131,
}

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))

def local_score(solution_df, submission_df):
    sol = solution_df.copy()
    sub = submission_df.copy()
    model_scores = []
    for mid in sorted(sol['model_id'].unique()):
        nt_raw = {1: [], 2: []}
        event_scores = []
        for evid in sorted(sol[sol['model_id']==mid]['event_id'].unique()):
            nt_scores = []
            for nt in [1, 2]:
                mask_s = (sol['model_id']==mid) & (sol['event_id']==evid) & (sol['node_type']==nt)
                mask_b = (sub['model_id']==mid) & (sub['event_id']==evid) & (sub['node_type']==nt)
                if mask_s.sum() == 0: continue
                s_df = sol[mask_s]; b_df = sub[mask_b]
                sd = STD_DEV_DICT.get((mid, nt), 1.0)
                node_std_rmses = []
                for nid in s_df['node_id'].unique():
                    nm_s = s_df[s_df['node_id']==nid]['water_level'].values
                    nm_b = b_df[b_df['node_id']==nid]['water_level'].values
                    if len(nm_s) > 1 and len(nm_s) == len(nm_b):
                        r = rmse(nm_s, nm_b)
                        node_std_rmses.append(r / sd)
                        nt_raw[nt].append(r)
                if node_std_rmses:
                    nt_scores.append(np.mean(node_std_rmses))
            if nt_scores:
                event_scores.append(np.mean(nt_scores))
        if event_scores:
            ms = np.mean(event_scores)
            model_scores.append(ms)
            print(f"  Model {mid}: Std RMSE = {ms:.6f} ({len(event_scores)} events)")
            for nt in [1, 2]:
                if nt_raw[nt]:
                    print(f"    NT{nt}: raw RMSE mean={np.mean(nt_raw[nt]):.6f}")
    final = np.mean(model_scores)
    print(f"  Overall: {final:.6f}")
    return final

In [4]:
# =========================================================
# UTILITIES
# =========================================================
def _free_memory():
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass

def ncol(col):
    col = str(col).strip().lower()
    col = col.replace('\u00b2', '2').replace('\u00b3', '3').replace('%', 'pct')
    col = re.sub(r'[^a-z0-9]+', '_', col)
    return re.sub(r'_+', '_', col).strip('_')

def read_csv(path):
    df = pd.read_csv(path)
    df.columns = [ncol(c) for c in df.columns]
    drop = [c for c in df.columns if c.startswith('unnamed')]
    if drop:
        df = df.drop(columns=drop)
    return df

def read_table(path):
    if str(path).endswith('.parquet'):
        df = pd.read_parquet(path)
    else:
        df = pd.read_csv(path)
    df.columns = [ncol(c) for c in df.columns]
    return df

def fcol(columns, names=(), tokens=(), required=True, label='col'):
    cols = list(columns)
    for n in names:
        if n in cols:
            return n
    for c in cols:
        if all(t in c for t in tokens):
            return c
    if required:
        raise KeyError(f"Cannot find {label} in {cols}")
    return None

def ensure_nid(df):
    c = fcol(df.columns, ('node_idx', 'node_id', 'id'), ('node',), False, 'nid')
    out = df.copy()
    if c is None:
        out['node_id'] = np.arange(len(out), dtype='int64')
    elif c != 'node_id':
        out = out.rename(columns={c: 'node_id'})
    out['node_id'] = out['node_id'].astype('int64')
    return out

def eid(d):
    return int(d.name.split('_')[1])

def mid_fn(d):
    return int(d.name.split('_')[1])

def event_dirs(split_dir):
    return sorted([d for d in split_dir.glob('event_*') if d.is_dir()], key=eid)

def dyn_path(event_dir, nt):
    return event_dir / f"{'1d' if nt == 1 else '2d'}_nodes_dynamic_all.csv"

def edge_dyn_path(event_dir, nt):
    return event_dir / f"{'1d' if nt == 1 else '2d'}_edges_dynamic_all.csv"

def find_file(model_dir, filename):
    for s in ('train', 'test'):
        p = model_dir / s / filename
        if p.exists():
            return p
    return None

def optimize_frame_dtypes(df, category_cols=('node_id_cat',)):
    out = df.copy()
    for c in out.columns:
        if c in category_cols and c in out.columns:
            out[c] = out[c].fillna('NA').astype('category')
            continue
        if pd.api.types.is_float_dtype(out[c]):
            out[c] = pd.to_numeric(out[c], downcast='float')
        elif pd.api.types.is_integer_dtype(out[c]):
            out[c] = pd.to_numeric(out[c], downcast='integer')
        elif out[c].dtype == 'object' and c.endswith('_cat'):
            out[c] = out[c].fillna('NA').astype('category')
    return out

def load_parquet_frames(paths, columns=None, category_cols=('node_id_cat',)):
    frames = []
    for p in paths:
        df = pd.read_parquet(p, columns=columns)
        frames.append(optimize_frame_dtypes(df, category_cols=category_cols))
    if not frames:
        return pd.DataFrame(columns=columns or [])
    out = pd.concat(frames, ignore_index=True)
    return optimize_frame_dtypes(out, category_cols=category_cols)

print("Utilities ready (RAM-optimized).")

Utilities ready (RAM-optimized).


In [5]:


# =========================================================
# STATIC NODE FEATURES
# =========================================================
STATIC_CACHE = {}

def load_static(model_id, node_type):
    k = (model_id, node_type)
    if k in STATIC_CACHE: return STATIC_CACHE[k]
    md = DATA_ROOT / f'Model_{model_id}'
    if node_type == 1:
        cands = ['1d_nodes_static.csv']
    else:
        cands = ['2d_nodes_static.csv', '2d_nodes_index.csv']
    for fn in cands:
        p = find_file(md, fn)
        if p:
            df = read_csv(p)
            df = ensure_nid(df)
            df = df.drop_duplicates('node_id').reset_index(drop=True)
            STATIC_CACHE[k] = df
            return df
    raise FileNotFoundError(f'Static not found for m{model_id} nt{node_type}')

print("Static loader ready.")

Static loader ready.


In [6]:
# =========================================================
# GRAPH FEATURES + MULTI-HOP + CENTRALITY + PIPE CAPACITY
# =========================================================
GRAPH_CACHE = {}
ADJ_CACHE = {}
EDGE_INDEX_CACHE = {}

def _count_upstream_total(upstream_dict, all_nids):
    cache = {}
    def _get_ancestors(nid, visited):
        if nid in cache: return cache[nid]
        if nid in visited: return set()
        visited.add(nid)
        result = set()
        for parent in upstream_dict.get(nid, []):
            result.add(parent)
            result.update(_get_ancestors(parent, visited))
        cache[nid] = result
        return result
    out = {}
    for nid in all_nids:
        out[int(nid)] = len(_get_ancestors(int(nid), set()))
    return out


def _compute_2hop_features(adj, node_feat_map, all_nids, feat_name):
    """Compute 2-hop neighbor aggregation for a given feature."""
    result = {}
    for nid in all_nids:
        nid = int(nid)
        hop1 = adj.get(nid, [])
        hop2_set = set()
        for n1 in hop1:
            for n2 in adj.get(n1, []):
                if n2 != nid and n2 not in hop1:
                    hop2_set.add(n2)
        if hop2_set:
            vals = [node_feat_map.get(n, np.nan) for n in hop2_set]
            vals = [v for v in vals if not np.isnan(v)]
            if vals:
                result[nid] = {
                    f'hop2_{feat_name}_mean': np.mean(vals),
                    f'hop2_{feat_name}_max': np.max(vals),
                    f'hop2_{feat_name}_min': np.min(vals),
                    f'hop2_count': len(hop2_set),
                }
    return result


def _pipe_capacity_manning(diameter, roughness, slope):
    """Manning's formula for full-pipe capacity: Q = (1/n) * A * R^(2/3) * S^(1/2).
    Circular pipe: A = pi*d^2/4, R = d/4.
    """
    if diameter <= 0 or roughness <= 0 or slope <= 0:
        return 0.0
    area = np.pi * diameter**2 / 4.0
    hyd_radius = diameter / 4.0
    return (1.0 / roughness) * area * hyd_radius**(2.0/3.0) * np.sqrt(slope)


def load_graph_feats(model_id, node_type):
    k = (model_id, node_type)
    if k in GRAPH_CACHE: return GRAPH_CACHE[k]
    md = DATA_ROOT / f'Model_{model_id}'
    prefix = '1d' if node_type == 1 else '2d'
    ei_path = find_file(md, f'{prefix}_edge_index.csv')

    if ei_path is None:
        print(f"  [WARN] {prefix}_edge_index.csv not found for Model_{model_id}")
        GRAPH_CACHE[k] = pd.DataFrame()
        return GRAPH_CACHE[k]

    ei = read_csv(ei_path)
    print(f"  Model {model_id} {prefix} edge_index: {len(ei)} edges, cols={list(ei.columns)}")

    cols = list(ei.columns)
    fc = fcol(cols, ('from_node','from_node_id','source'), ('from',), False, 'from')
    tc = fcol(cols, ('to_node','to_node_id','target'), ('to',), False, 'to')
    if fc is None or tc is None: fc, tc = cols[0], cols[1]

    EDGE_INDEX_CACHE[k] = (fc, tc, ei)

    adj = defaultdict(set)
    downstream = defaultdict(list)
    upstream = defaultdict(list)
    for _, r in ei.iterrows():
        u, v = int(r[fc]), int(r[tc])
        adj[u].add(v); adj[v].add(u)
        downstream[u].append(v)
        upstream[v].append(u)
    adj = {k_: list(v) for k_, v in adj.items()}
    ADJ_CACHE[k] = adj

    # Load edge static features
    es_path = find_file(md, f'{prefix}_edges_static.csv')
    edge_static = read_csv(es_path) if es_path else None
    if edge_static is not None:
        print(f"    Edge static: {len(edge_static)} edges, cols={list(edge_static.columns)}")

    efmap = {}
    if edge_static is not None and len(ei) == len(edge_static):
        for i in range(len(ei)):
            u, v = int(ei.iloc[i][fc]), int(ei.iloc[i][tc])
            efmap[(u, v)] = edge_static.iloc[i].to_dict()

    # 1D-2D connections
    conn = find_file(md, '1d2d_connections.csv')
    conn_map = defaultdict(list)
    conn_1d_to_2d = defaultdict(list)
    if conn:
        cdf = read_csv(conn)
        cc = list(cdf.columns)
        c1 = fcol(cc, ('1d_node_id','node_1d'), ('1d',), False, '1d')
        c2 = fcol(cc, ('2d_node_id','node_2d'), ('2d',), False, '2d')
        if c1 is None or c2 is None: c1, c2 = cc[0], cc[1]
        for _, r in cdf.iterrows():
            n1, n2 = int(r[c1]), int(r[c2])
            conn_1d_to_2d[n1].append(n2)
            if node_type == 1: conn_map[n1].append(n2)
            else: conn_map[n2].append(n1)

    static = load_static(model_id, node_type)
    all_nids = static['node_id'].unique()

    # Transitive upstream count (1D only)
    upstream_total = {}
    if node_type == 1:
        upstream_total = _count_upstream_total(upstream, all_nids)
        print(f"    Upstream total: max={max(upstream_total.values()) if upstream_total else 0}")

    # Elevation maps for gradients + 2-hop
    elev_map = {}
    if node_type == 2:
        elev_c = fcol(static.columns, ('elevation','min_elevation','centroid_elevation'),
                      ('elev',), False, 'elev')
        if elev_c:
            elev_map = dict(zip(static['node_id'].astype(int), static[elev_c].astype(float)))
    elif node_type == 1:
        inv_c = fcol(static.columns, ('invert_elevation',), ('invert',), False, 'inv')
        if inv_c:
            elev_map = dict(zip(static['node_id'].astype(int), static[inv_c].astype(float)))

    # Enhanced coupling: elevation difference 1D<->2D
    coupling_elev_diff = {}
    if node_type == 1 and conn_1d_to_2d:
        s2d = load_static(model_id, 2)
        elev_c_2d = fcol(s2d.columns, ('elevation','min_elevation','centroid_elevation'),
                         ('elev',), False, 'elev')
        inv_c = fcol(static.columns, ('invert_elevation',), ('invert',), False, 'inv')
        if elev_c_2d and inv_c:
            elev_2d_map = dict(zip(s2d['node_id'].astype(int), s2d[elev_c_2d].astype(float)))
            inv_map = dict(zip(static['node_id'].astype(int), static[inv_c].astype(float)))
            for n1d, n2d_list in conn_1d_to_2d.items():
                e2d_vals = [elev_2d_map.get(n2, np.nan) for n2 in n2d_list]
                e2d_clean = [v for v in e2d_vals if not np.isnan(v)]
                inv_val = inv_map.get(n1d, np.nan)
                if e2d_clean and not np.isnan(inv_val):
                    coupling_elev_diff[n1d] = np.mean(e2d_clean) - inv_val

    # v5.5: Graph centrality via NetworkX (fast for small 1D graphs, ok for 2D)
    centrality_pr = {}
    centrality_deg = {}
    try:
        import networkx as nx
        G = nx.Graph()
        for _, r in ei.iterrows():
            G.add_edge(int(r[fc]), int(r[tc]))
        centrality_pr = nx.pagerank(G, max_iter=100, tol=1e-4)
        centrality_deg = nx.degree_centrality(G)
        print(f"    Centrality: PageRank + degree for {G.number_of_nodes()} nodes")
    except Exception as e:
        print(f"    [WARN] centrality failed: {e}")

    # v5.5: 2-hop features
    hop2_feats = {}
    if elev_map:
        hop2_feats = _compute_2hop_features(adj, elev_map, all_nids, 'elev')
        print(f"    2-hop elev features: {len(hop2_feats)} nodes")

    # v5.5: Pipe capacity (1D only, Manning's formula)
    pipe_cap_map = {}
    if node_type == 1 and efmap:
        node_cap_in = defaultdict(list)
        node_cap_out = defaultdict(list)
        for (u, v), ef in efmap.items():
            diam = float(ef.get('diameter', 0) or 0)
            rough = float(ef.get('roughness', 0) or 0)
            slp = abs(float(ef.get('slope', 0) or 0))
            cap = _pipe_capacity_manning(diam, rough, slp)
            node_cap_out[u].append(cap)
            node_cap_in[v].append(cap)
        for nid in all_nids:
            nid = int(nid)
            caps_in = node_cap_in.get(nid, [])
            caps_out = node_cap_out.get(nid, [])
            all_caps = caps_in + caps_out
            if all_caps:
                pipe_cap_map[nid] = {
                    'pipe_cap_total_in': sum(caps_in),
                    'pipe_cap_total_out': sum(caps_out),
                    'pipe_cap_max': max(all_caps),
                    'pipe_cap_mean': np.mean(all_caps),
                }
        print(f"    Pipe capacity: {len(pipe_cap_map)} nodes")

    recs = []
    for nid in all_nids:
        nid = int(nid)
        r = {'node_id': nid, 'degree': len(adj.get(nid, []))}

        # Centrality
        r['pagerank'] = centrality_pr.get(nid, 0.0)
        r['degree_centrality'] = centrality_deg.get(nid, 0.0)

        # 2-hop features
        if nid in hop2_feats:
            r.update(hop2_feats[nid])

        if node_type == 1:
            up = upstream.get(nid, [])
            dn = downstream.get(nid, [])
            r['n_upstream'] = len(up)
            r['n_downstream'] = len(dn)
            r['upstream_total'] = upstream_total.get(nid, 0)

            # Aggregate upstream/downstream edge static features
            up_feats = [efmap.get((u, nid), {}) for u in up]
            dn_feats = [efmap.get((nid, v), {}) for v in dn]
            for pname, flist in [('up', up_feats), ('dn', dn_feats)]:
                if flist:
                    for fname in list(flist[0].keys())[:8]:
                        vals = []
                        for f in flist:
                            try: vals.append(float(f.get(fname, 0)))
                            except: pass
                        if vals:
                            r[f'{pname}_{fname}_mean'] = np.mean(vals)

            conn_nodes = conn_map.get(nid, [])
            r['has_2d_conn'] = int(len(conn_nodes) > 0)
            r['n_2d_conn'] = len(conn_nodes)
            r['coupling_elev_diff'] = coupling_elev_diff.get(nid, np.nan)

            # Pipe capacity
            if nid in pipe_cap_map:
                r.update(pipe_cap_map[nid])

        else:  # 2D
            conn_nodes = conn_map.get(nid, [])
            r['has_1d_conn'] = int(len(conn_nodes) > 0)
            r['n_1d_conn'] = len(conn_nodes)
            nbrs = adj.get(nid, [])
            if nbrs and elev_map:
                own_elev = elev_map.get(nid, np.nan)
                nbr_elevs = [elev_map.get(n, np.nan) for n in nbrs]
                nbr_elevs_clean = [e for e in nbr_elevs if not np.isnan(e)]
                if nbr_elevs_clean:
                    r['nbr_mean_elev'] = np.mean(nbr_elevs_clean)
                    r['nbr_min_elev'] = np.min(nbr_elevs_clean)
                    if not np.isnan(own_elev):
                        r['elev_gradient_mean'] = own_elev - np.mean(nbr_elevs_clean)
                        r['elev_gradient_max'] = own_elev - np.min(nbr_elevs_clean)

            # v5: aggregate 2D static edge features to nodes
            if efmap:
                nbr_edge_feats = []
                for nb in nbrs:
                    ef = efmap.get((nid, nb)) or efmap.get((nb, nid))
                    if ef: nbr_edge_feats.append(ef)
                if nbr_edge_feats:
                    for fname in list(nbr_edge_feats[0].keys())[:6]:
                        vals = []
                        for f in nbr_edge_feats:
                            try: vals.append(float(f.get(fname, 0)))
                            except: pass
                        if vals:
                            r[f'nbr_{fname}_mean'] = np.mean(vals)
        recs.append(r)

    result = pd.DataFrame(recs)
    result['node_id'] = result['node_id'].astype('int64')
    print(f"    Graph features: {len(result)} nodes, {len(result.columns)-1} features")
    GRAPH_CACHE[k] = result
    return result


# --- v6 Experimental: Spectral graph embeddings ---
SPECTRAL_CACHE = {}

def compute_spectral_embeddings(model_id, node_type):
    """Compute Laplacian eigenvector embeddings from graph structure."""
    if not USE_SPECTRAL_EMBED:
        return pd.DataFrame()
    k = (model_id, node_type)
    if k in SPECTRAL_CACHE:
        return SPECTRAL_CACHE[k]

    try:
        ei = load_edge_index(model_id, node_type)
        if ei is None or len(ei) == 0:
            SPECTRAL_CACHE[k] = pd.DataFrame()
            return pd.DataFrame()

        src_col = fcol(ei.columns, ('source','from','src'), ('node','id'), label='src')
        dst_col = fcol(ei.columns, ('target','to','dst','dest'), ('node','id'), label='dst')
        if src_col is None or dst_col is None:
            SPECTRAL_CACHE[k] = pd.DataFrame()
            return pd.DataFrame()

        nodes = sorted(set(ei[src_col].tolist() + ei[dst_col].tolist()))
        n = len(nodes)
        if n < SPECTRAL_DIM + 2:
            SPECTRAL_CACHE[k] = pd.DataFrame()
            return pd.DataFrame()

        nid_map = {v: i for i, v in enumerate(nodes)}
        rows = [nid_map[v] for v in ei[src_col]]
        cols = [nid_map[v] for v in ei[dst_col]]
        # Symmetric adjacency
        rows_full = rows + cols
        cols_full = cols + rows
        vals = np.ones(len(rows_full), dtype='float32')
        A = csr_matrix((vals, (rows_full, cols_full)), shape=(n, n))
        # Remove self-loops, ensure binary
        A.setdiag(0)
        A.eliminate_zeros()
        A.data[:] = 1.0

        # Laplacian: L = D - A
        deg = np.array(A.sum(axis=1)).flatten()
        D = csr_matrix((deg, (range(n), range(n))), shape=(n, n))
        L = D - A

        # Normalized Laplacian for stability
        deg_inv_sqrt = np.zeros(n, dtype='float64')
        mask = deg > 0
        deg_inv_sqrt[mask] = 1.0 / np.sqrt(deg[mask])
        D_inv_sqrt = csr_matrix((deg_inv_sqrt, (range(n), range(n))), shape=(n, n))
        L_norm = D_inv_sqrt @ L @ D_inv_sqrt

        # Smallest non-trivial eigenvectors
        n_eig = min(SPECTRAL_DIM + 1, n - 1)
        eigenvalues, eigenvectors = eigsh(L_norm.astype('float64'), k=n_eig, which='SM')

        # Skip first eigenvector (constant), take next SPECTRAL_DIM
        emb = eigenvectors[:, 1:SPECTRAL_DIM+1].astype('float32')
        result = pd.DataFrame(emb, columns=[f'spectral_{i}' for i in range(emb.shape[1])])
        result['node_id'] = nodes
        SPECTRAL_CACHE[k] = result
        print(f"    Spectral embeddings: {emb.shape[1]} dims for {n} nodes")
        return result

    except Exception as e:
        print(f"    Spectral embeddings failed: {e}")
        SPECTRAL_CACHE[k] = pd.DataFrame()
        return pd.DataFrame()

print("Graph feature loader ready (v5.5: +multi-hop, +centrality, +pipe capacity).")

Graph feature loader ready (v5.5: +multi-hop, +centrality, +pipe capacity).


In [7]:
# =========================================================
# DIST TO NEAREST DRAIN (for 2D nodes)
# =========================================================
DIST_DRAIN_CACHE = {}

def load_dist_to_drain(model_id):
    if model_id in DIST_DRAIN_CACHE: return DIST_DRAIN_CACHE[model_id]
    try:
        s1d = load_static(model_id, 1)
        s2d = load_static(model_id, 2)
        x1 = fcol(s1d.columns, ('position_x',), ('position','x'), False, 'x')
        y1 = fcol(s1d.columns, ('position_y',), ('position','y'), False, 'y')
        x2 = fcol(s2d.columns, ('position_x',), ('position','x'), False, 'x')
        y2 = fcol(s2d.columns, ('position_y',), ('position','y'), False, 'y')
        if not all([x1, y1, x2, y2]):
            DIST_DRAIN_CACHE[model_id] = pd.DataFrame()
            return DIST_DRAIN_CACHE[model_id]
        from scipy.spatial import cKDTree
        tree = cKDTree(s1d[[x1, y1]].values.astype(float))
        dists, _ = tree.query(s2d[[x2, y2]].values.astype(float), k=min(3, len(s1d)))
        result = pd.DataFrame({'node_id': s2d['node_id'].values,
                               'dist_nearest_drain': dists[:, 0] if dists.ndim > 1 else dists})
        if dists.ndim > 1 and dists.shape[1] >= 2: result['dist_2nd_drain'] = dists[:, 1]
        if dists.ndim > 1 and dists.shape[1] >= 3: result['dist_3rd_drain'] = dists[:, 2]
        result['node_id'] = result['node_id'].astype('int64')
        print(f"  dist_nearest_drain Model_{model_id}: {len(result)} nodes")
        DIST_DRAIN_CACHE[model_id] = result
        return result
    except Exception as e:
        print(f"  [WARN] dist_to_drain failed: {e}")
        DIST_DRAIN_CACHE[model_id] = pd.DataFrame()
        return DIST_DRAIN_CACHE[model_id]

print("Dist-to-drain ready.")

Dist-to-drain ready.


In [8]:
# =========================================================
# RAIN FEATURES (v6: +future rain, +duration, +dry spell, +quantiles)
# =========================================================
RAIN_CACHE = {}
TIMESTEP_DURATION_CACHE = {}

def load_timestep_durations(model_id, event_dir):
    """v5.5: Load actual timestep durations from timesteps.csv."""
    k = (model_id, eid(event_dir))
    if k in TIMESTEP_DURATION_CACHE: return TIMESTEP_DURATION_CACHE[k]
    ts_path = event_dir / 'timesteps.csv'
    if not ts_path.exists():
        TIMESTEP_DURATION_CACHE[k] = None
        return None
    try:
        ts_df = read_csv(ts_path)
        # Expect columns like: timestep, datetime or time or seconds
        ts_col = fcol(ts_df.columns, ('timestep',), ('time','step'), False, 'ts')
        time_col = fcol(ts_df.columns, ('datetime','time','seconds','elapsed'),
                        ('time',), False, 'time')
        if ts_col is None or time_col is None:
            TIMESTEP_DURATION_CACHE[k] = None
            return None
        ts_df = ts_df.sort_values(ts_col).reset_index(drop=True)
        # Try to parse as seconds or datetime
        try:
            time_vals = pd.to_numeric(ts_df[time_col])
            durations = time_vals.diff().fillna(time_vals.iloc[0] if len(time_vals) > 0 else 0)
        except (ValueError, TypeError):
            try:
                time_vals = pd.to_datetime(ts_df[time_col])
                durations = time_vals.diff().dt.total_seconds().fillna(0)
            except:
                TIMESTEP_DURATION_CACHE[k] = None
                return None
        result = pd.DataFrame({
            'timestep': ts_df[ts_col].astype('int64'),
            'step_duration': durations.astype('float32'),
            'elapsed_time': time_vals.astype('float64') if pd.api.types.is_numeric_dtype(time_vals) else durations.cumsum().astype('float32'),
        })
        TIMESTEP_DURATION_CACHE[k] = result
        return result
    except Exception:
        TIMESTEP_DURATION_CACHE[k] = None
        return None


def load_rain(model_id, event_dir):
    k = (model_id, eid(event_dir))
    if k in RAIN_CACHE: return RAIN_CACHE[k]
    path_2d = dyn_path(event_dir, 2)
    df = read_csv(path_2d)
    df = ensure_nid(df)
    ts_col = fcol(df.columns, ('timestep',), ('time','step'), label='timestep')
    rain_col = fcol(df.columns, ('rainfall',), ('rain',), label='rainfall')
    rain_ts = df.groupby(ts_col)[rain_col].mean().reset_index()
    rain_ts.columns = ['timestep', 'rain_global']
    rain_ts = rain_ts.sort_values('timestep').reset_index(drop=True)
    rain = rain_ts['rain_global'].astype('float64')

    out = pd.DataFrame({
        'timestep': rain_ts['timestep'].astype('int64'),
        'rain_global': rain,
        'rain_lag1': rain.shift(1).fillna(0),
        'rain_lag2': rain.shift(2).fillna(0),
        'rain_lag3': rain.shift(3).fillna(0),
        'rain_roll3': rain.rolling(3, min_periods=1).sum(),
        'rain_roll6': rain.rolling(6, min_periods=1).sum(),
        'rain_roll12': rain.rolling(12, min_periods=1).sum(),
        'rain_roll24': rain.rolling(24, min_periods=1).sum(),
        'rain_cumsum': rain.cumsum(),
        'rain_delta': rain - rain.shift(1).fillna(0),
        'is_raining': (rain > 0).astype('int8'),
        'rain_intensity_peak': rain.expanding().max(),
    })

    for i in range(1,500):
        out[f'dodepch1k_lag_{i}'] = rain.shift(i).fillna(0)

    out['rain_accel'] = out['rain_delta'] - out['rain_delta'].shift(1).fillna(0)
    for alpha in [0.1, 0.05, 0.02, 0.01]:
        out[f'rain_ema_{int(1/alpha)}'] = rain.ewm(alpha=alpha, adjust=False).mean()

    total = rain.sum()
    out['rain_cum_ratio'] = (rain.cumsum() / total) if total > 0 else 0

    peak = rain.max()
    peak_ts = rain_ts.loc[rain.idxmax(), 'timestep'] if peak > 0 else 0
    max_ts = rain_ts['timestep'].max()
    out['event_total_rain'] = total
    out['event_peak_rain'] = peak
    out['time_since_rain_peak'] = out['timestep'] - peak_ts
    out['frac_event'] = out['timestep'] / (max_ts + 1)
    out['rain_phase_rising'] = (out['rain_delta'] > 0).astype('int8')
    out['rain_phase_falling'] = ((out['rain_delta'] < 0) & (rain > 0)).astype('int8')

    # v5.5: Rain duration & dry spell
    is_rain = (rain > 0).values
    rain_dur = np.zeros(len(rain), dtype='int32')
    dry_spell = np.zeros(len(rain), dtype='int32')
    cnt_rain, cnt_dry = 0, 0
    for i in range(len(rain)):
        if is_rain[i]:
            cnt_rain += 1
            cnt_dry = 0
        else:
            cnt_dry += 1
            cnt_rain = 0
        rain_dur[i] = cnt_rain
        dry_spell[i] = cnt_dry
    out['rain_duration'] = rain_dur
    out['dry_spell'] = dry_spell

    # v5.5: Intensity quantiles (expanding)
    out['rain_q75'] = rain.expanding().quantile(0.75)
    out['rain_q90'] = rain.expanding().quantile(0.90)

    # v5.5: Rain rate of change (smoothed)
    out['rain_roc_smooth'] = rain.rolling(3, min_periods=1).mean().diff().fillna(0)


    # v6: Future rain features (forecast fully available — organizer confirmed)
    rain_vals = rain.values
    n_rain = len(rain_vals)
    for h in [1, 3, 5, 10]:
        shifted = rain.shift(-h).fillna(0)
        out[f'rain_future_{h}'] = shifted.astype('float32')

    # Future rolling sums: sum of rain in next w steps (excluding current)
    for w in [5, 10]:
        cs = rain.cumsum().values
        total_cs = cs[-1]
        fsum = np.zeros(n_rain, dtype='float32')
        for i in range(n_rain):
            end_idx = min(i + w, n_rain - 1)
            fsum[i] = cs[end_idx] - cs[i]
        out[f'rain_future_sum{w}'] = fsum

    # Future max: max rain in next w steps
    for w in [5, 10]:
        fmax = np.zeros(n_rain, dtype='float32')
        for i in range(n_rain):
            end_idx = min(i + w + 1, n_rain)
            if i + 1 < end_idx:
                fmax[i] = rain_vals[i+1:end_idx].max()
        out[f'rain_future_max{w}'] = fmax

    # Rain remaining after current timestep
    out['rain_remaining'] = (total - rain.cumsum()).astype('float32')

    # Rain future acceleration (will it intensify or weaken?)
    rain_future_1 = rain.shift(-1).fillna(0)
    rain_future_3 = rain.shift(-3).fillna(0)
    out['rain_future_trend'] = (rain_future_3 - rain).astype('float32')

    # v5.5: Timestep durations
    ts_dur = load_timestep_durations(model_id, event_dir)
    if ts_dur is not None:
        out = out.merge(ts_dur, on='timestep', how='left')

    RAIN_CACHE[k] = out
    return out

print("Rain features ready (v5.5: +duration, +dry_spell, +quantiles, +timestep_duration).")

Rain features ready (v5.5: +duration, +dry_spell, +quantiles, +timestep_duration).


In [9]:
# =========================================================
# WARMUP FEATURES + TRENDS + NEIGHBOR WARMUP
# + v5 NEW: INLET FLOW, WATER VOLUME, EDGE DYNAMICS
# =========================================================
WARMUP_CACHE = {}
WARMUP_DYN_CACHE = {}

def load_warmup(model_id, split, event_dir, node_type):
    key = (model_id, eid(event_dir), node_type, split)
    if key in WARMUP_CACHE: return WARMUP_CACHE[key]
    df = read_csv(dyn_path(event_dir, node_type))
    df = ensure_nid(df)
    ts_col = fcol(df.columns, ('timestep',), ('time','step'), label='timestep')
    wl_col = fcol(df.columns, ('water_level',), ('water','level'), label='water_level')
    warm = df[['node_id', ts_col, wl_col]].copy()
    warm.columns = ['node_id', 'timestep', 'water_level']
    warm['timestep'] = warm['timestep'].astype('int64')
    warm['water_level'] = warm['water_level'].astype('float64')
    warm = warm.sort_values(['node_id', 'timestep']).reset_index(drop=True)
    warm = warm[warm['timestep'] < WARMUP_STEPS].copy()
    warm['wi'] = warm.groupby('node_id').cumcount()
    wide = warm.pivot(index='node_id', columns='wi', values='water_level')
    wide.columns = [f'water_level_{int(c)}' for c in wide.columns]
    wide = wide.reset_index()
    for i in range(WARMUP_STEPS):
        c = f'water_level_{i}'
        if c not in wide.columns: wide[c] = np.nan
    WARMUP_CACHE[key] = wide
    return wide


def _agg_warmup_series(vals):
    clean = vals[~np.isnan(vals)]
    if len(clean) == 0:
        return np.nan, np.nan, np.nan, np.nan
    return np.mean(clean), np.max(clean), np.min(clean), clean[-1]


def load_warmup_dynamics(model_id, split, event_dir, node_type):
    key = (model_id, eid(event_dir), node_type, split)
    if key in WARMUP_DYN_CACHE: return WARMUP_DYN_CACHE[key]

    result_recs = {}

    # --- Node-level dynamic features (inlet_flow / water_volume) ---
    node_dyn = read_csv(dyn_path(event_dir, node_type))
    node_dyn = ensure_nid(node_dyn)
    ts_col = fcol(node_dyn.columns, ('timestep',), ('time','step'), label='timestep')
    node_dyn['timestep'] = node_dyn[ts_col].astype('int64')
    warmup_nodes = node_dyn[node_dyn['timestep'] < WARMUP_STEPS].copy()

    if node_type == 1:
        inlet_col = fcol(warmup_nodes.columns, ('1d_node_inlet_flow','inlet_flow'),
                         ('inlet','flow'), False, 'inlet_flow')
        if inlet_col:
            for nid, grp in warmup_nodes.groupby('node_id'):
                vals = grp[inlet_col].astype('float64').values
                mn, mx, mi, last = _agg_warmup_series(vals)
                result_recs.setdefault(int(nid), {})['inlet_flow_mean'] = mn
                result_recs.setdefault(int(nid), {})['inlet_flow_max'] = mx
                result_recs.setdefault(int(nid), {})['inlet_flow_min'] = mi
                result_recs.setdefault(int(nid), {})['inlet_flow_last'] = last
                abs_vals = np.abs(vals[~np.isnan(vals)])
                result_recs[int(nid)]['inlet_flow_abs_mean'] = np.mean(abs_vals) if len(abs_vals) else np.nan
                result_recs[int(nid)]['inlet_flow_abs_max'] = np.max(abs_vals) if len(abs_vals) else np.nan
    else:
        vol_col = fcol(warmup_nodes.columns, ('water_volume',), ('volume',), False, 'volume')
        if vol_col:
            for nid, grp in warmup_nodes.groupby('node_id'):
                vals = grp[vol_col].astype('float64').values
                mn, mx, mi, last = _agg_warmup_series(vals)
                result_recs.setdefault(int(nid), {})['wvol_mean'] = mn
                result_recs.setdefault(int(nid), {})['wvol_max'] = mx
                result_recs.setdefault(int(nid), {})['wvol_last'] = last
                clean = vals[~np.isnan(vals)]
                if len(clean) >= 2:
                    result_recs[int(nid)]['wvol_trend'] = clean[-1] - clean[0]
                else:
                    result_recs[int(nid)]['wvol_trend'] = np.nan

    del warmup_nodes, node_dyn

    # --- Edge-level dynamic features aggregated to nodes ---
    edge_path = edge_dyn_path(event_dir, node_type)
    k_ei = (model_id, node_type)
    if edge_path.exists() and k_ei in EDGE_INDEX_CACHE:
        fc_name, tc_name, ei_df = EDGE_INDEX_CACHE[k_ei]
        edge_dyn = read_csv(edge_path)
        ts_e = fcol(edge_dyn.columns, ('timestep',), ('time','step'), label='timestep')
        edge_dyn['timestep'] = edge_dyn[ts_e].astype('int64')
        warmup_edges = edge_dyn[edge_dyn['timestep'] < WARMUP_STEPS].copy()
        del edge_dyn

        if node_type == 1:
            flow_col = fcol(warmup_edges.columns, ('1d_edge_flow','edge_flow'),
                            ('edge','flow'), False, 'flow')
            vel_col = fcol(warmup_edges.columns, ('1d_edge_velocity','edge_velocity'),
                           ('velocity',), False, 'velocity')
        else:
            flow_col = fcol(warmup_edges.columns, ('2d_flow',), ('flow',), False, 'flow')
            vel_col = fcol(warmup_edges.columns, ('2d_velocity',), ('velocity',), False, 'velocity')

        if flow_col or vel_col:
            eidx_col = fcol(warmup_edges.columns, ('edge_idx','edge_id','link_id'),
                            ('edge',), False, 'edge_idx')

            edge_to_nodes = {}
            for i in range(len(ei_df)):
                edge_to_nodes[i] = (int(ei_df.iloc[i][fc_name]), int(ei_df.iloc[i][tc_name]))

            node_in_flows, node_out_flows = defaultdict(list), defaultdict(list)
            node_in_vels, node_out_vels = defaultdict(list), defaultdict(list)

            if eidx_col:
                edge_groups = warmup_edges.groupby(eidx_col)
            else:
                n_edges = len(ei_df)
                warmup_edges['_edge_idx'] = warmup_edges.groupby('timestep').cumcount()
                edge_groups = warmup_edges.groupby('_edge_idx')
                eidx_col = '_edge_idx'

            for edge_idx, grp in edge_groups:
                edge_idx = int(edge_idx)
                if edge_idx not in edge_to_nodes: continue
                from_n, to_n = edge_to_nodes[edge_idx]

                if flow_col and flow_col in grp.columns:
                    fvals = grp[flow_col].astype('float64').values
                    fmean = np.nanmean(fvals)
                    fabs_mean = np.nanmean(np.abs(fvals))
                    fabs_max = np.nanmax(np.abs(fvals))
                    node_out_flows[from_n].append((fmean, fabs_mean, fabs_max))
                    node_in_flows[to_n].append((fmean, fabs_mean, fabs_max))

                if vel_col and vel_col in grp.columns:
                    vvals = grp[vel_col].astype('float64').values
                    vmean = np.nanmean(vvals)
                    vabs_mean = np.nanmean(np.abs(vvals))
                    vabs_max = np.nanmax(np.abs(vvals))
                    node_out_vels[from_n].append((vmean, vabs_mean, vabs_max))
                    node_in_vels[to_n].append((vmean, vabs_mean, vabs_max))

            del warmup_edges

            prefix = '1d' if node_type == 1 else '2d'
            all_nids = set()
            for d in [node_in_flows, node_out_flows, node_in_vels, node_out_vels]:
                all_nids.update(d.keys())

            for nid in all_nids:
                rec = result_recs.setdefault(int(nid), {})

                if nid in node_in_flows:
                    in_f = node_in_flows[nid]
                    rec[f'{prefix}_in_flow_mean'] = np.mean([x[0] for x in in_f])
                    rec[f'{prefix}_in_flow_abs_mean'] = np.mean([x[1] for x in in_f])
                    rec[f'{prefix}_in_flow_abs_max'] = np.max([x[2] for x in in_f])

                if nid in node_out_flows:
                    out_f = node_out_flows[nid]
                    rec[f'{prefix}_out_flow_mean'] = np.mean([x[0] for x in out_f])
                    rec[f'{prefix}_out_flow_abs_mean'] = np.mean([x[1] for x in out_f])
                    rec[f'{prefix}_out_flow_abs_max'] = np.max([x[2] for x in out_f])

                if nid in node_in_vels:
                    in_v = node_in_vels[nid]
                    rec[f'{prefix}_in_vel_mean'] = np.mean([x[0] for x in in_v])
                    rec[f'{prefix}_in_vel_abs_mean'] = np.mean([x[1] for x in in_v])
                    rec[f'{prefix}_in_vel_abs_max'] = np.max([x[2] for x in in_v])

                if nid in node_out_vels:
                    out_v = node_out_vels[nid]
                    rec[f'{prefix}_out_vel_mean'] = np.mean([x[0] for x in out_v])
                    rec[f'{prefix}_out_vel_abs_mean'] = np.mean([x[1] for x in out_v])
                    rec[f'{prefix}_out_vel_abs_max'] = np.max([x[2] for x in out_v])

                if node_type == 1:
                    in_mean = rec.get(f'{prefix}_in_flow_mean', 0) or 0
                    out_mean = rec.get(f'{prefix}_out_flow_mean', 0) or 0
                    rec[f'{prefix}_net_flow'] = in_mean - out_mean

    if result_recs:
        rows = [{'node_id': nid, **feats} for nid, feats in result_recs.items()]
        result = pd.DataFrame(rows)
        result['node_id'] = result['node_id'].astype('int64')
    else:
        result = pd.DataFrame()

    WARMUP_DYN_CACHE[key] = result
    return result


def compute_warmup_trends(warmup_wide, static_df, node_type):
    wl_cols = sorted([c for c in warmup_wide.columns if c.startswith('water_level_')])
    trend = pd.DataFrame({'node_id': warmup_wide['node_id'].astype('int64')})
    if not wl_cols: return trend
    vals = warmup_wide[wl_cols].values.astype('float64')
    trend['warmup_mean'] = np.nanmean(vals, axis=1)
    trend['warmup_std'] = np.nanstd(vals, axis=1)
    trend['warmup_range'] = np.nanmax(vals, axis=1) - np.nanmin(vals, axis=1)
    trend['warmup_last'] = vals[:, -1]
    if vals.shape[1] >= 2:
        trend['warmup_last_delta'] = vals[:, -1] - vals[:, -2]
    if vals.shape[1] >= 3:
        trend['warmup_accel'] = (vals[:, -1] - vals[:, -2]) - (vals[:, -2] - vals[:, -3])
    x = np.arange(vals.shape[1], dtype='float64')
    xm = x.mean(); xvar = ((x - xm)**2).sum()
    if xvar > 0:
        trend['warmup_trend'] = np.nansum((x[None,:] - xm) * vals, axis=1) / xvar
    if node_type == 1:
        inv_c = fcol(static_df.columns, ('invert_elevation',), ('invert',), False, 'inv')
        surf_c = fcol(static_df.columns, ('surface_elevation',), ('surface',), False, 'surf')
        if inv_c and surf_c:
            merged = trend.merge(static_df[['node_id', inv_c, surf_c]], on='node_id', how='left')
            inv_v = merged[inv_c].fillna(0).values
            surf_v = merged[surf_c].fillna(0).values
            pipe_d = surf_v - inv_v
            trend['fill_ratio'] = np.where(pipe_d > 0, (trend['warmup_last'].values - inv_v) / pipe_d, 0)
            trend['is_surcharged'] = (trend['warmup_last'].values > surf_v).astype('int8')
    return trend


def compute_neighbor_warmup(model_id, node_type, warmup_wide):
    adj = ADJ_CACHE.get((model_id, node_type), {})
    if not adj or 'water_level_9' not in warmup_wide.columns:
        return pd.DataFrame()
    wl9_map = dict(zip(warmup_wide['node_id'].astype(int),
                       warmup_wide['water_level_9'].astype('float64')))
    records = []
    for nid in warmup_wide['node_id'].astype(int):
        nid = int(nid)
        nbrs = adj.get(nid, [])
        own_wl9 = wl9_map.get(nid, np.nan)
        if nbrs:
            nbr_wl9 = [wl9_map.get(n, np.nan) for n in nbrs]
            nbr_mean = np.nanmean(nbr_wl9)
            records.append({'node_id': nid, 'nbr_mean_wl9': nbr_mean,
                           'nbr_max_wl9': np.nanmax(nbr_wl9),
                           'gradient_wl9': (own_wl9 - nbr_mean) if pd.notna(own_wl9) and pd.notna(nbr_mean) else np.nan})
        else:
            records.append({'node_id': nid, 'nbr_mean_wl9': np.nan,
                           'nbr_max_wl9': np.nan, 'gradient_wl9': np.nan})
    return pd.DataFrame(records)

print("Warmup features ready (v5: +inlet_flow, +water_volume, +edge dynamics).")

Warmup features ready (v5: +inlet_flow, +water_volume, +edge dynamics).


In [10]:
# =========================================================
# BUILD SUPERVISED FRAME (v6: +spectral embeddings, +aux targets)
# =========================================================
def build_supervised_event_frame(model_id, split, event_dir, node_type, include_target=True):
    df_dyn = read_csv(dyn_path(event_dir, node_type))
    df_dyn = ensure_nid(df_dyn)
    ts_col = fcol(df_dyn.columns, ('timestep',), ('time','step'), label='timestep')
    wl_col = fcol(df_dyn.columns, ('water_level',), ('water','level'),
                  required=include_target, label='water_level')
    rain_col = fcol(df_dyn.columns, ('rainfall',), ('rain',), required=False, label='rainfall')

    keep = [ts_col, 'node_id']
    if include_target and wl_col: keep.append(wl_col)
    if node_type == 2 and rain_col: keep.append(rain_col)

    df = df_dyn[keep].copy()
    del df_dyn
    renames = {ts_col: 'timestep'}
    if include_target and wl_col: renames[wl_col] = 'water_level'
    if node_type == 2 and rain_col: renames[rain_col] = 'rain_local'
    df = df.rename(columns=renames)
    df['timestep'] = df['timestep'].astype('int64')
    df = df[df['timestep'] >= MIN_PRED_TIMESTEP].copy()

    df['model_id'] = model_id
    df['event_id'] = eid(event_dir)
    df['node_type'] = node_type

    # Static features
    static = load_static(model_id, node_type)
    df = df.merge(static, on='node_id', how='left')

    # Rain features
    rain = load_rain(model_id, event_dir)
    df = df.merge(rain, on='timestep', how='left')

    # Warmup water levels
    warmup = load_warmup(model_id, split, event_dir, node_type)
    df = df.merge(warmup, on='node_id', how='left')

    # Graph features
    gf = load_graph_feats(model_id, node_type)
    if len(gf) > 0 and 'node_id' in gf.columns:
        df = df.merge(gf, on='node_id', how='left')

    # Distance to drain (2D)
    if node_type == 2:
        dd = load_dist_to_drain(model_id)
        if len(dd) > 0 and 'node_id' in dd.columns:
            df = df.merge(dd, on='node_id', how='left')

    # Warmup trends
    trend = compute_warmup_trends(warmup, static, node_type)
    df = df.merge(trend, on='node_id', how='left')

    # Neighbor warmup
    nbr = compute_neighbor_warmup(model_id, node_type, warmup)
    if len(nbr) > 0:
        df = df.merge(nbr, on='node_id', how='left')

    # Warmup dynamics (inlet_flow, water_volume, edge flow/velocity)
    wdyn = load_warmup_dynamics(model_id, split, event_dir, node_type)
    if len(wdyn) > 0 and 'node_id' in wdyn.columns:
        df = df.merge(wdyn, on='node_id', how='left')

    if 'rain_local' not in df.columns:
        df['rain_local'] = df.get('rain_global', 0)

    # Interaction features (1D only)
    if 'upstream_total' in df.columns:
        df['upstream_x_rain_cumsum'] = df['upstream_total'] * df['rain_cumsum']
        df['upstream_x_rain_ema_50'] = df['upstream_total'] * df.get('rain_ema_50', 0)

    if 'inlet_flow_abs_mean' in df.columns:
        df['inlet_x_rain_cumsum'] = df['inlet_flow_abs_mean'] * df['rain_cumsum']

    if 'fill_ratio' in df.columns:
        df['fill_x_rain_cumsum'] = df['fill_ratio'] * df['rain_cumsum']

    if 'pipe_cap_total_in' in df.columns:
        df['cap_x_rain_cumsum'] = df['pipe_cap_total_in'] * df['rain_cumsum']
        df['cap_x_fill'] = df['pipe_cap_total_in'] * df.get('fill_ratio', 0)

    # v6: Spectral graph embeddings
    se = compute_spectral_embeddings(model_id, node_type)
    if len(se) > 0 and 'node_id' in se.columns:
        df = df.merge(se, on='node_id', how='left')


    # v6: Auxiliary target predictions (inlet_flow)
    if USE_AUX_TARGETS and 'AUX_MODELS' in dir() and (model_id, node_type) in AUX_MODELS:
        aux_lgb, feat_aux = AUX_MODELS[(model_id, node_type)]
        avail = [c for c in feat_aux if c in df.columns]
        if len(avail) == len(feat_aux):
            df['pred_inlet_flow'] = aux_lgb.predict(df[feat_aux]).astype('float32')
            df['pred_inlet_x_rain'] = df['pred_inlet_flow'] * df.get('rain_cumsum', 0)
        else:
            df['pred_inlet_flow'] = np.float32(0)
            df['pred_inlet_x_rain'] = np.float32(0)
    elif USE_AUX_TARGETS:
        df['pred_inlet_flow'] = np.float32(0)
        df['pred_inlet_x_rain'] = np.float32(0)

    df['node_id_cat'] = df['node_id'].astype(str)
    df['log_timestep'] = np.log1p(df['timestep'].astype('float32'))

    if include_target and USE_TARGET_DELTA and 'water_level_9' in df.columns:
        df['wl_delta_target'] = df['water_level'].astype('float64') - df['water_level_9'].astype('float64')

    for c in df.columns:
        if c == 'node_id_cat': continue
        if df[c].dtype == 'object':
            conv = pd.to_numeric(df[c], errors='coerce')
            if conv.notna().mean() >= 0.95: df[c] = conv
            else: df[c] = df[c].fillna('NA').astype(str)

    df = df.sort_values(['node_id', 'timestep']).reset_index(drop=True)
    if include_target and 'water_level' in df.columns:
        df['water_level'] = pd.to_numeric(df['water_level'], downcast='float')
    if include_target and 'wl_delta_target' in df.columns:
        df['wl_delta_target'] = pd.to_numeric(df['wl_delta_target'], downcast='float')

    df = optimize_frame_dtypes(df)
    return df


def get_features(df):
    excluded = set(KEY_COLS + [TARGET_COL, 'event_id', 'wl_delta_target'])
    feat = [c for c in df.columns if c not in excluded]
    cat = [c for c in feat if df[c].dtype == 'object' or c.endswith('_cat')]
    return feat, cat




print("Frame builder ready (RAM-optimized, no event frame cache).")

Frame builder ready (RAM-optimized, no event frame cache).


In [11]:
def optimize_frame_dtypes(df, category_cols=('node_id_cat',)):
    out = df.copy()
    for c in out.columns:
        s = out[c]

        is_cat_target = (
            (c in category_cols)
            or (s.dtype == 'object' and c.endswith('_cat'))
            or pd.api.types.is_categorical_dtype(s)
        )

        if is_cat_target:
            if pd.api.types.is_categorical_dtype(s):
                if s.isna().any():
                    if pd.api.types.is_numeric_dtype(s.cat.categories):
                        sentinel = -1
                    else:
                        sentinel = 'NA'
                    if sentinel not in list(s.cat.categories):
                        s = s.cat.add_categories([sentinel])
                    s = s.fillna(sentinel)
                out[c] = s
            else:
                if pd.api.types.is_numeric_dtype(s):
                    s = pd.to_numeric(s, errors='coerce')
                    if s.isna().any():
                        s = s.fillna(-1)
                    out[c] = pd.Series(s, index=out.index).astype('category')
                else:
                    out[c] = s.astype('string').fillna('NA').astype('category')
            continue

        if pd.api.types.is_float_dtype(s):
            out[c] = pd.to_numeric(s, downcast='float')
        elif pd.api.types.is_integer_dtype(s):
            out[c] = pd.to_numeric(s, downcast='integer')

    return out

In [12]:
# =========================================================
# DATASET INDEXING & SPLIT
# =========================================================
assert DATA_ROOT.exists()
model_dirs = sorted(DATA_ROOT.glob('Model_*'))
assert len(model_dirs) == 2

train_events, test_events = {}, {}
for d in model_dirs:
    m = mid_fn(d)
    train_events[m] = event_dirs(d / 'train')
    test_events[m] = event_dirs(d / 'test')
    print(f"Model {m}: {len(train_events[m])} train, {len(test_events[m])} test")

train_pool, valid_pool = {}, {}
for m in sorted(train_events):
    tr, va = train_test_split(train_events[m], test_size=VALID_EVENTS_PER_MODEL,
                               random_state=RANDOM_SEED + m, shuffle=True)
    train_pool[m] = sorted(tr, key=eid)
    valid_pool[m] = sorted(va, key=eid)
    print(f"  Split: {len(train_pool[m])} train, {len(valid_pool[m])} valid")

Model 1: 68 train, 29 test
Model 2: 69 train, 30 test
  Split: 53 train, 15 valid
  Split: 54 train, 15 valid


In [13]:
import shutil
shutil.copytree('/kaggle/input/datasets/andrewrwerewrew/hmmmmmmmm-676767/feature_store_m2_nt2','/kaggle/working/feature_store_m2_nt2')

'/kaggle/working/feature_store_m2_nt2'

In [14]:
# =========================================================
# MATERIALIZE FEATURE STORE + LOAD ONLY NEEDED SPLITS FOR (model_id=2, node_type=2)
# =========================================================
TARGET_MODEL_ID = 2
TARGET_NODE_TYPE = 2

print(f"Preparing RAM-optimized data for Model {TARGET_MODEL_ID}, NodeType {TARGET_NODE_TYPE}")
FEATURE_STORE_DIR.mkdir(parents=True, exist_ok=True)

def _clear_event_caches():
    for name in [
        'RAIN_CACHE', 'WARMUP_CACHE', 'WARMUP_DYN_CACHE', 'TIMESTEP_DURATION_CACHE',
        'EVENT_FRAME_CACHE'
    ]:
        if name in globals():
            globals()[name].clear()
    if 'SPECTRAL_CACHE' in globals():
        SPECTRAL_CACHE.clear()
    _free_memory()

def feature_store_path(model_id, split, event_id, node_type):
    return FEATURE_STORE_DIR / f"m{model_id}_{split}_e{event_id}_nt{node_type}.parquet"

def materialize_split(model_id, split, event_dirs_list, node_type, include_target):
    records = []
    for i, ed in enumerate(event_dirs_list, 1):
        p = feature_store_path(model_id, split, eid(ed), node_type)
        if FORCE_REBUILD_FEATURE_STORE or (not p.exists()):
            _clear_event_caches()
            df_evt = build_supervised_event_frame(
                model_id, split, ed, node_type, include_target=include_target
            )
            df_evt = optimize_frame_dtypes(df_evt)
            df_evt.to_parquet(p, index=False)
            rows = len(df_evt)
            cols = len(df_evt.columns)
            del df_evt
            _free_memory()
        else:
            meta = pd.read_parquet(p, columns=['node_id'])
            rows = len(meta)
            cols = None
            del meta
        records.append({
            'model_id': model_id,
            'split': split,
            'event_id': eid(ed),
            'node_type': node_type,
            'include_target': include_target,
            'path': str(p),
            'rows': rows,
            'cols': cols,
        })
        if i % 5 == 0 or i == len(event_dirs_list):
            print(f"  {split}: materialized {i}/{len(event_dirs_list)} events")
    _clear_event_caches()
    return pd.DataFrame(records)

train_manifest = materialize_split(TARGET_MODEL_ID, 'train', train_pool[TARGET_MODEL_ID], TARGET_NODE_TYPE, include_target=True)
eval_manifest = materialize_split(TARGET_MODEL_ID, 'train', valid_pool[TARGET_MODEL_ID], TARGET_NODE_TYPE, include_target=True)
test_manifest = materialize_split(TARGET_MODEL_ID, 'test', test_events[TARGET_MODEL_ID], TARGET_NODE_TYPE, include_target=False)

sample_train_path = Path(train_manifest.iloc[0]['path'])
sample_train_df = pd.read_parquet(sample_train_path)
feature_cols, cat_cols = get_features(sample_train_df)
target_col = 'wl_delta_target' if (USE_TARGET_DELTA and 'wl_delta_target' in sample_train_df.columns) else TARGET_COL

ordered_cols = []
for c in KEY_COLS + [TARGET_COL, 'wl_delta_target'] + feature_cols:
    if c in sample_train_df.columns and c not in ordered_cols:
        ordered_cols.append(c)
del sample_train_df
_free_memory()

Preparing RAM-optimized data for Model 2, NodeType 2
  train: materialized 5/54 events
  train: materialized 10/54 events
  train: materialized 15/54 events
  train: materialized 20/54 events
  train: materialized 25/54 events
  train: materialized 30/54 events
  train: materialized 35/54 events
  train: materialized 40/54 events
  train: materialized 45/54 events
  train: materialized 50/54 events
  train: materialized 54/54 events
  train: materialized 5/15 events
  train: materialized 10/15 events
  train: materialized 15/15 events
  test: materialized 5/30 events
  test: materialized 10/30 events
  test: materialized 15/30 events
  test: materialized 20/30 events
  test: materialized 25/30 events
  test: materialized 30/30 events


In [15]:
import gc
import pandas as pd
import polars as pl


def load_manifest_df(
    manifest_df,
    columns=None,
    category_cols=(),
    event_ids=None,
    verbose=True,
    return_lazy=True,
    to_pandas=False,
):
    """
    RAM-бережная загрузка parquet через polars.

    Что делает:
    - не читает parquet в pandas по одному
    - не делает pandas concat
    - строит lazy scan для каждого parquet
    - объединяет через pl.concat(..., how="diagonal_relaxed"),
      чтобы пережить несовпадающие схемы (Int8 vs Int16 и т.п.)
    - по умолчанию возвращает LazyFrame
    """

    if event_ids is not None:
        event_ids = set(event_ids)
        sub = manifest_df.loc[
            manifest_df["event_id"].isin(event_ids), ["path"]
        ].reset_index(drop=True)
    else:
        sub = manifest_df.loc[:, ["path"]].reset_index(drop=True)

    paths = sub["path"].tolist()
    n_total = len(paths)

    if n_total == 0:
        if return_lazy:
            return pl.LazyFrame()
        if to_pandas:
            return pd.DataFrame(columns=columns or [])
        return pl.DataFrame(schema={c: pl.Null for c in (columns or [])})

    if verbose:
        print(f"Building lazy scan over {n_total} parquet files")

    # Читаем только схемы файлов (дёшево по памяти)
    schemas = [pl.read_parquet_schema(path) for path in paths]

    if columns is None:
        # union всех колонок с сохранением порядка первого появления
        seen = set()
        use_cols = []
        for sch in schemas:
            for c in sch.keys():
                if c not in seen:
                    seen.add(c)
                    use_cols.append(c)
    else:
        # только реально существующие в хотя бы одном файле
        use_cols = [c for c in columns if any(c in sch for sch in schemas)]

    if not use_cols:
        if return_lazy:
            return pl.LazyFrame()
        if to_pandas:
            return pd.DataFrame(columns=columns or [])
        return pl.DataFrame()

    lazy_parts = []

    for path, sch in zip(paths, schemas):
        present_cols = [c for c in use_cols if c in sch]
        if not present_cols:
            continue

        try:
            lf_part = pl.scan_parquet(
                path,
                low_memory=True,
                cache=False,
                cast_options=pl.ScanCastOptions(
                    integer_cast="upcast",
                    float_cast="upcast",
                ),
            )
        except TypeError:
            # fallback для старых версий polars
            lf_part = pl.scan_parquet(
                path,
                low_memory=True,
                cache=False,
            )

        lf_part = lf_part.select(present_cols)
        lazy_parts.append(lf_part)

    if not lazy_parts:
        if return_lazy:
            return pl.LazyFrame()
        if to_pandas:
            return pd.DataFrame(columns=use_cols)
        return pl.DataFrame(schema={c: pl.Null for c in use_cols})

    # Ключевой фикс:
    # diagonal_relaxed -> объединяет разные наборы колонок
    # и приводит конфликтующие типы к общему supertype
    lf = pl.concat(
        lazy_parts,
        how="diagonal_relaxed",
        rechunk=False,
        parallel=True,
    )

    # Восстанавливаем желаемый порядок колонок
    lf = lf.select(use_cols)

    # Категориальные колонки приводим лениво
    cat_exprs = []
    for c in category_cols:
        if c in use_cols:
            cat_exprs.append(
                pl.col(c)
                .cast(pl.Utf8, strict=False)
                .fill_null("NA")
                .cast(pl.Categorical)
                .alias(c)
            )

    if cat_exprs:
        lf = lf.with_columns(cat_exprs)

    if return_lazy:
        return lf

    if verbose:
        print("Collecting materialized DataFrame from lazy scan...")

    try:
        df_pl = lf.collect(engine="streaming")
    except TypeError:
        # fallback для старых версий
        df_pl = lf.collect(streaming=True)

    gc.collect()
    if "_free_memory" in globals():
        _free_memory()

    if to_pandas:
        if verbose:
            print("Converting polars -> pandas...")
        df_pd = df_pl.to_pandas()
        del df_pl
        gc.collect()
        if "_free_memory" in globals():
            _free_memory()
        return df_pd

    return df_pl

In [16]:
train_df = load_manifest_df(train_manifest, columns=ordered_cols, category_cols=tuple(cat_cols))
eval_df = load_manifest_df(eval_manifest, columns=ordered_cols, category_cols=tuple(cat_cols))
test_df = load_manifest_df(test_manifest, columns=ordered_cols, category_cols=tuple(cat_cols))

Building lazy scan over 54 parquet files
Building lazy scan over 15 parquet files
Building lazy scan over 30 parquet files


In [17]:
train_df = pl.concat([train_df, eval_df])

In [18]:
train_df = train_df.drop([f'dodepch1k_lag_{i}' for i in range(1,500)])
test_df = test_df.drop([f'dodepch1k_lag_{i}' for i in range(1,500)])

In [19]:
train_df = train_df.collect().to_pandas()
test_df = test_df.collect().to_pandas()

In [20]:
train_df = pd.concat([train_df,pd.read_parquet('/kaggle/input/bobritobandito/dop_feats_train_full.parquet')],axis=1)
test_df = pd.concat([test_df,pd.read_parquet('/kaggle/input/bobritobandito/dop_feats_test.parquet')],axis=1)

In [21]:
train_df['ridge_p'] = np.load('/kaggle/input/bobritobandito/full_ridge_train.npy')
test_df['ridge_p'] = np.load('/kaggle/input/bobritobandito/full_ridge_test.npy')

In [22]:
dataset_info = {
    'model_id': TARGET_MODEL_ID,
    'node_type': TARGET_NODE_TYPE,
    'target_col': target_col,
    'feature_cols': feature_cols,
    'cat_cols': cat_cols,
    'n_features': len(feature_cols),
    'train_rows': int(train_manifest['rows'].sum()),
    'eval_rows': int(eval_manifest['rows'].sum()),
    'test_rows': int(test_manifest['rows'].sum()),
    'train_paths': train_manifest['path'].tolist(),
    'eval_paths': eval_manifest['path'].tolist(),
    'test_paths': test_manifest['path'].tolist(),
}

In [23]:
SEED = 58

CB_PARAMS = {
    'loss_function': 'RMSE',
    'iterations': 12_000, 
    'max_depth': 8, 
    'learning_rate': 0.16,
    'border_count': 254,
    'eval_metric': 'RMSE',
    'task_type': 'GPU',
    'random_seed': SEED,
    'verbose': 500,
}


LGB_PARAMS = {
    'objective': 'regression',
     'n_estimators': 20_000, 
    'max_depth': 8, 
    'learning_rate': 0.16,
    'reg_lambda': 3, 
    'subsample': 0.8, 
    'colsample_bytree': 0.8,
    'metric': 'rmse',
    'verbosity': -1,
    'n_jobs': -1,
    'random_state': SEED,
    'device': 'gpu',
}


XGB_PARAMS = {
        'objective': 'reg:squarederror',
        'n_estimators': 12_000,
        'max_depth': 8,
        'learning_rate': 0.16,
        'reg_lambda': 3,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'eval_metric': 'rmse',
        'device': 'cuda',
        'tree_method': 'hist',
        'predictor': 'gpu_predictor',
        'random_state': SEED,
}

In [24]:
# =========================================================
# MODEL SETUP FOR (model_id=2, node_type=2)
# =========================================================
m = TARGET_MODEL_ID
nt = TARGET_NODE_TYPE
cb_params = CB_PARAMS
lgb_params = LGB_PARAMS
xgb_params = XGB_PARAMS

feat = dataset_info['feature_cols']
feat = dataset_info['feature_cols']
feat = [x for x in feat if 'dodepch1k' not in x]
feat += ['giga_fft']
feat += ['ridge_p']
feat += [f'giga_fft_{q}' for q in [0.8, 0.9, 0.95, 0.99]]

cat = dataset_info['cat_cols']
feat_lgb = [f for f in feat if f != 'node_id_cat']
feat_xgb = [f for f in feat if f != 'node_id_cat']
use_delta = (dataset_info['target_col'] == 'wl_delta_target')
target = dataset_info['target_col']
sd = STD_DEV_DICT[(m, nt)]
specs = {}
valid_preds = {}
sol_parts = []
sub_parts_cb, sub_parts_lgb, sub_parts_xgb = [], [], []

In [25]:
cb_refit_model = CatBoostRegressor(**cb_params)
cb_refit_model.fit(
    train_df[feat], train_df[target],
    cat_features=cat,
    verbose=cb_params.get('verbose', 500),
)

0:	learn: 2.3156780	total: 870ms	remaining: 2h 54m 3s
500:	learn: 0.1133690	total: 4m 59s	remaining: 1h 54m 27s
1000:	learn: 0.0881197	total: 10m	remaining: 1h 49m 59s
1500:	learn: 0.0761539	total: 15m 1s	remaining: 1h 45m 3s
2000:	learn: 0.0684602	total: 20m 2s	remaining: 1h 40m 11s
2500:	learn: 0.0629197	total: 25m 5s	remaining: 1h 35m 17s
3000:	learn: 0.0589583	total: 30m 6s	remaining: 1h 30m 18s
3500:	learn: 0.0557593	total: 35m 7s	remaining: 1h 25m 16s
4000:	learn: 0.0530911	total: 40m 7s	remaining: 1h 20m 13s
4500:	learn: 0.0509078	total: 45m 8s	remaining: 1h 15m 12s
5000:	learn: 0.0489699	total: 50m 7s	remaining: 1h 10m 8s
5500:	learn: 0.0473443	total: 55m 6s	remaining: 1h 5m 6s
6000:	learn: 0.0458696	total: 1h 7s	remaining: 1h 6s
6500:	learn: 0.0445641	total: 1h 5m 8s	remaining: 55m 6s
7000:	learn: 0.0433809	total: 1h 10m 8s	remaining: 50m 5s
7500:	learn: 0.0422989	total: 1h 15m 9s	remaining: 45m 4s
8000:	learn: 0.0413129	total: 1h 20m 8s	remaining: 40m 3s
8500:	learn: 0.040379

In [26]:
lgb_refit_model = lgb.LGBMRegressor(**lgb_params)
lgb_refit_model.fit(train_df[feat_lgb], train_df[target])

1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


LGBMRegressor(colsample_bytree=0.8, device='gpu', learning_rate=0.16,
              max_depth=8, metric='rmse', n_estimators=20000, n_jobs=-1,
              objective='regression', random_state=58, reg_lambda=3,
              subsample=0.8, verbosity=-1)

In [27]:
xgb_refit_model = xgb.XGBRegressor(**xgb_params)
xgb_refit_model.fit(train_df[feat_xgb], train_df[target], verbose=300)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device='cuda', early_stopping_rounds=None,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.16, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=12000,
             n_jobs=None, num_parallel_tree=None, ...)

In [28]:
cb_test_pred = cb_refit_model.predict(test_df[feat])
lgb_test_pred = lgb_refit_model.predict(test_df[feat_lgb])
xgb_test_pred = xgb_refit_model.predict(test_df[feat_xgb])
ens_test_pred = cb_test_pred * 0.35 + lgb_test_pred * 0.2 + xgb_test_pred * 0.45

test_pred_df = test_df[KEY_COLS].copy()
test_pred_df['pred_cb'] = np.asarray(cb_test_pred, dtype='float32')
test_pred_df['pred_lgb'] = np.asarray(lgb_test_pred, dtype='float32')
test_pred_df['pred_xgb'] = np.asarray(xgb_test_pred, dtype='float32')
test_pred_df[TARGET_COL] = np.asarray(ens_test_pred, dtype='float32')

submission_m2_nt2_df = test_pred_df[KEY_COLS + [TARGET_COL]].copy()
submission_m2_nt2_df.to_parquet('submission_m2_nt2_cb_lgb_xgb_ensemble.parquet', index=False)
test_pred_df.to_parquet('submission_m2_nt2_cb_lgb_xgb_components.parquet', index=False)

print(f"test_df memory={test_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"Saved: submission_m2_nt2_cb_lgb_xgb_ensemble.parquet ({len(submission_m2_nt2_df):,} rows)")
print(f"Saved: submission_m2_nt2_cb_lgb_xgb_components.parquet ({len(test_pred_df):,} rows)")

del test_df
_free_memory()

test_df memory=14310.7 MB
Saved: submission_m2_nt2_cb_lgb_xgb_ensemble.parquet (30,875,418 rows)
Saved: submission_m2_nt2_cb_lgb_xgb_components.parquet (30,875,418 rows)
